# Datamine MODSPLIT Process Example

This notebook demonstrates how to configure and run the **`modsplit`** process wrapper in `dmstudio`.

## Process Description

## MODSPLIT

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

**MODSPLIT** splits a Datamine cell/subcell model using an input wireframe to create a new model that is constrained by the wireframe.

Either a solid wireframe with multiple closed volumes or a digital terrain model wireframe can be used. The volume of the wireframe is accurately represented in the split model. An optional output model **FULLMOD** may be specified. This is a model that covers the same total volume as the input model but with cell splitting to model the wireframe volumes. The input wireframe may contain multiple wireframes that are identified using the * **ZONE** field in the wireframe triangle file. The * **ZONE** values will then be included in the output model files.

**MODSPLIT** is useful for situations in which you have a mine design or mineable shapes in the form of wireframe and you wish to quickly represent those within a reserve block model. You can choose to output a model containing just the cells constrained by the wireframe shapes (**MODELOUT**) or the whole model (**FULLMOD**). The resulting model can then be used in processes such as **[TONGRAD](<tongrad.md>)** to produce reserves tables.

The input model may be rotated. If the input model &**PROTO** is a Rotated Model, as defined using the **[PROTOM](<protom.md>)** process, then the coordinates of the input points in the &**WIREPT** file are automatically transformed internally to the rotated model coordinate system for processing.

If both the input model file and the input wireframe triangle file include field ZONE then to avoid confusion in the output model files field **ZONE** from the input model file will be renamed as _MD_ZN_ _n in the output model files. n is an index from 0-9. The first replacement name will be _MD_ZN_0_. If it already exists then _MD_ZN_1_ will be used and so on.

### Input Files:

* **modelin** (Block Model):
  Input model file.
  Required=Yes

* **wiretr** (Wireframe):
  Input wireframe triangle file used to split the model cells.
  Required=Yes

* **wirept** (Wireframe):
  Input wireframe point file.
  Required=Yes

### Output Files:

* **modelout** (Input):
  Block Model
  Required=No

* **fullmod** (Input):
  Block Model
  Required=No

### Fields:

* **zone** (Undefined : WIRETR):
  Name of zone field, if any, in input wireframe with multiple zones. If the field is
  specified then it is created in the output model file with the corresponding zone value
  as defined by the wireframe. The field can be either numeric or 4 character alpha. If
  not specified and a field **ZONE** exists in **WIRETR** then it will automatically be
  used.
  Default=Undefined
  Required=No

* **mined** (Numeric :):
  Optional field to be added to the output full model. Cells constrained by the
  wireframes will have a value of 1, cells outside or not constrained by the wireframes
  will have a value of zero
  Default=Undefined
  Required=No

* **density** (Numeric : MODELIN):
  Density field in the input model file to be used when averaging grade fields. If this
  is not set and a **DENSITY** field exists in the model file it will be used. Otherwise
  Density is specified using the **DENSITY** parameter.
  Default=DENSITY
  Required=No

### Parameters:

* **modltype**:
  Type of wireframe model to be filled; one of the following options, with default of (1)
  :- =1 : solid 3d, interior to be filled with cells. =2 : solid 3d, exterior to be filled
  with cells. =3 : surface, cells to be filled below (for XY), to south (for XZ), or to
  west (for YZ). =4 : surface, cells to be filled above (for XY), to north (for XZ), or to
  east (for YZ). =5 : fill between two surfaces with cells.. =6 : two surfaces, cells to
  be filled above upper surface and below lower surface.
  Range=1,6
  Values=1,2,3,4,5,6
  Default=1
  Required=Yes

* **zone**:
  Zone code to be inserted into output model ZONE field if field does not exist in input
  wireframe model.
  Range=
  Values=
  Default=0
  Required=No

* **maxdip**:
  Reference gradient used to calculate the degree of cell splitting as described in the
  Full Description (0).
  Range=0,90
  Values=
  Default=0
  Required=No

* **splits**:
  Maximum amount of splitting to be allowed (3). =0 : no splitting: parent cell. =1 : 1

* **split** (2 x 2 subcells. =2 : 2 splits: 4 x 4 subcells. =3 : 3 splits: 8 x 8 subcells.):
  Range=1,3
  Values=1,2,3
  Default=3
  Required=No

* **plane**:
  Optional alpha parameter defining orientation 'XY', 'XZ', or 'YZ', of plane in which
  subcell splitting is to be performed. Care must be taken in selection of the plane to be
  used if the ends of the wireframe have not been linked, as the wireframe model is then
  partially 'hollow' when viewed from certain directions.
  Range=
  Values=XY, XZ, YZ
  Default=XY
  Required=No

* **xsubcell**:
  Number of subcells in X direction (1). Max 100. Only used if SPLITS=0.
  Range=1,100
  Values=
  Default=1
  Required=No

* **ysubcell**:
  Number of subcells in Y direction (1). Max 100. Only used if SPLITS=0.
  Range=1,100
  Values=
  Default=1
  Required=No

* **zsubcell**:
  Number of subcells in Z direction (1). Max 100. Only used if SPLITS=0.
  Range=1,100
  Values=
  Default=1
  Required=No

* **resol**:
  Defines boundary resolution in direction perpendicular to plane of filling. =0 :
  precise boundary location. =N : boundary rounded to nearest 1/Nth of parent cell size.
  Range=0,N
  Values=
  Default=0
  Required=No

* **density**:
  Default Density value to be used when averaging grades. This is used if there is no
  **DENSITY** field in the input block model or if Density values in the model are absent.
  Range=
  Values=
  Default=1
  Required=No

* **defgrade**:
  The calculation of average grades within cells that are not constrained by the
  wireframes is controlled by the @**DEFGRADE** parameter.
  Range=
  Values=
  Default=
  Required=No

* **usezone**:
  If the wireframe triangle file includes field ZONE then by default MODSPLIT will use it
  for zone control unless a different zone field is explicitly selected. Parameter
  @**USEZONE** has been introduced to allow field **ZONE** to exist in the triangle file
  but not used for zone control.
  Range=
  Values=
  Default=
  Required=No

* **tolernce**:
  Defines the smallest cell that will be included in **OUT**. Defined as a factor

## **XINC** , **YINC** , **ZINC**.

  Range=undefined
  Values=undefined
  Default=0.001
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('modsplit')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute modsplit
print("Running modsplit...")
dm_cmd.modsplit(
    modelin_i='t_mod1',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    modltype_p='required_val',  # required
    # modelout_o='t_modsplit_out',  # optional
    # fullmod_o='t_modsplit_out',  # optional
    # zone_f='optional',  # optional
    # mined_f='optional',  # optional
    # density_f='optional',  # optional
    # zone_p=0,  # optional
    # maxdip_p=0,  # optional
    # splits_p=3,  # optional
    # plane_p='XY',  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # resol_p=0,  # optional
    # density_p=1,  # optional
    # defgrade_p='optional',  # optional
    # usezone_p='optional',  # optional
    # tolernce_p=0.001,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("modsplit execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_modsplit_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")